In [1]:
from langchain_core.chat_history import InMemoryChatMessageHistory  # 메모리에 대화 기록을 저장하는 클래스
from langchain_core.runnables.history import RunnableWithMessageHistory  # 메시지 기록을 활용해 실행 가능한 래퍼wrapper 클래스
from langchain_openai import ChatOpenAI  # 오픈AI 모델을 사용하는 랭체인 챗봇 클래스
from langchain_core.messages import HumanMessage

model = ChatOpenAI(model="gpt-5.6-luna")

# 세션별 대화 기록을 저장할 딕셔너리
store = {}

# 세션 ID에 따라 대화 기록을 가져오는 함수
def get_session_history(session_id: str):
    # 만약 해당 세션 ID가 store에 없으면, 새로 생성해 추가함
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()  # 메모리에 대화 기록을 저장하는 객체 생성
    return store[session_id]  # 해당 세션의 대화 기록을 반환

# 모델 실행 시 대화 기록을 함께 전달하는 래퍼 객체 생성
with_message_history = RunnableWithMessageHistory(model, get_session_history)

d:\GitHub\learn_do_it_llm_agent\ch08\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


# LangChain Deprecated

`RunnableWithMessageHistory`는 **LangChain에서 Deprecated 예정**이다.
→ 현재는 **LangGraph의 Persistence 기능** 사용을 권장한다.

[LangChain Python Reference](https://reference.langchain.com/python/langchain-core/runnables/history/RunnableWithMessageHistory)에는 아직 반영되지 않았지만,
[LangChain 공식 GitHub 소스](https://github.com/langchain-ai/langchain/blob/master/libs/core/langchain_core/runnables/history.py)에는 **2.0.0에서 제거 예정**으로 명시되어 있다.

## LangChain

LangChain은 **LLM 앱/Agent를 만들기 위한 프레임워크**이며, OpenAI API와는 별개의 기술이다.

| 기술           | 역할                          |
| ------------ | --------------------------- |
| Python / C++ | 프로그래밍 언어                    |
| OpenAI API   | OpenAI 서버와 통신하는 API         |
| LangChain    | LLM 앱/Agent 프레임워크           |
| LangGraph    | Agent의 상태·흐름·Persistence 관리 |


In [2]:
config = {"configurable": {"session_id": "abc2"}}  # 세션 ID를 설정하는 config 객체 생성

response = with_message_history.invoke(
    [HumanMessage(content="안녕? 난 이성용이야.")],
    config=config,
)

print(response.content)


안녕하세요, 이성용님! 만나서 반가워요. 무엇을 도와드릴까요?


In [3]:
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

이성용님이에요.


In [4]:
config = {"configurable": {"session_id": "abc3"}}

response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

response.content

'아직 이름을 알려주지 않으셨어요.'

session id를 바꿨기 때문에 저장되지 않음

In [5]:
config = {"configurable": {"session_id": "abc2"}}
for r in with_message_history.stream(
    [HumanMessage(content = "내가 어느 나라 사람인지 맞춰보고, 그 나라의 문화에 대해 말해봐")],
    config=config,
):
    print(r.content, end="|")

|이|름|과| 한국|어|를| 쓰|신|다는| 점|을| 보면| **|대한|민국| 사람|일| 가능|성이| 높|다고| 추|측|**|할| 수|는| 있지만|,| 이름|이나| 언|어|만|으로| 국|적|을| 단|정|할| 수|는| 없|어요|.

|만|약| 한국|을| 말씀|하|신| 거|라|면|,| 한국| 문화|에는| 다음|과| 같은| 특징|이| 있습니다|.

|-| **|가|족|과| 공동|체|를| 중|시|하는| 경|향|**|:| 가족| 간| 유|대|와| 소|속|감을| 중요|하게| 여|기는| 편|입니다|.
|-| **|유|교|적| 영향|**|:| 연|장|자|에| 대한| 예|의|,| 존|댓|말|,| 나|이|와| 직|급|에| 따른| 호|칭| 등이| 일|상|에| 남|아| 있습니다|.
|-| **|음|식| 문화|**|:| 김|치|,| 비|빔|밥|,| 불|고|기|처럼| 함께| 나|누|어| 먹|는| 음식|이| 많|고|,| 식|사|에서| 반|찬|과| 국|을| 곁|들이|는| 특징|이| 있습니다|.
|-| **|명|절|과| 전|통|**|:| 설|날|과| 추|석|에| 가족|이| 모|이고|,| 차|례|나| 세|배| 같은| 풍|습|을| 경험|하기|도| 합니다|.
|-| **|대|중|문화|**|:| K|-pop|,| 드|라마|,| 영화|,| 웹|툰| 등이| 세계|적으로| 큰| 영향|력을| 갖|고| 있습니다|.
|-| **|빠|른| 현대|화|와| 전|통|의| 공|존|**|:| 첨|단| 기술|과| 도시| 문화|가| 발|달|한| 동시에| 한|옥|,| 전|통|공|연|,| 지역| 축|제| 같은| 전|통|문화|도| 이어|지고| 있습니다|.

|다|만| 한국|인|이라고| 해서| 모두| 같은| 가치|관|이나| 생활|방|식을| 가진| 것은| 아니|며|,| 세|대|·|지역|·|개|인|에| 따라| 문화|가| 다양|합니다|.||||